# Vulcan Project: SQL Analysis

In the previous notebook (`clean_data.ipynb`) we cleaned the four raw tables: `customers`, `vehicles`, `service_records` and `complaints`. The cleaned versions were saved as CSV files.

In this notebook we will:

1. Load the cleaned tables into a PostgreSQL database called `vulcan`.
2. Check that the data was loaded correctly (row counts and data types).
3. Use SQL queries to analyse the data and find useful insights about the customers, vehicles, services and complaints.

The queries are run from Python (using `sqlalchemy`), so that the results come back as DataFrames and can be used for visualisation later.

First let us download the required tools that will let us connect with `PostgeSQL`

In [2]:
#%pip install sqlalchemy psycopg2-binary

## Loading the cleaned data

First we import the libraries and read the four cleaned CSV files into DataFrames. The `phone` column is read as a string so that the leading zeros are not lost, and the date columns are parsed as dates so that they are stored as proper date types in the database.

In [1]:
import pandas as pd
from sqlalchemy import create_engine

In [2]:
customers = pd.read_csv('customers_clean.csv', dtype={'phone': str}, parse_dates=['signup_date'])
vehicles = pd.read_csv('vehicles_clean.csv', parse_dates=['purchase_date'])
service = pd.read_csv('service_clean.csv', parse_dates=['service_date'])
complaints = pd.read_csv('complaints_clean.csv', parse_dates=['complaint_date'])

In [3]:
print(customers.shape, vehicles.shape, service.shape, complaints.shape)

(400, 6) (550, 9) (2200, 7) (350, 6)


The files are loaded and the shapes look fine.

## Connecting to PostgreSQL

Now we create a connection to the `VULCAN` database using `sqlalchemy`. The password is typed in when the cell runs (using `getpass`), so it is not saved inside the notebook and does not get uploaded to GitHub by mistake.

In [4]:
from getpass import getpass
from sqlalchemy.engine import URL

In [5]:
password = getpass('Enter PostgreSQL password: ')

url = URL.create(
    drivername='postgresql+psycopg2',
    username='postgres',
    password=password,
    host='localhost',
    port=5432,
    database='VULCAN'
)

engine = create_engine(url)

In [6]:
with engine.connect() as conn:
    print('Connected successfully')

Connected successfully


## Loading the tables into the database

Each DataFrame is written to the database as a table using `to_sql`. The tables are named `customers`, `vehicles`, `service_records` and `complaints`. `if_exists='replace'` means that if we run the cell again, the old table is replaced instead of adding duplicate rows.

In [7]:
customers.to_sql('customers', engine, if_exists='replace', index=False)
vehicles.to_sql('vehicles', engine, if_exists='replace', index=False)
service.to_sql('service_records', engine, if_exists='replace', index=False)
complaints.to_sql('complaints', engine, if_exists='replace', index=False)

350

Lets confirm that the tables are in the database and that the row counts match the DataFrames.

In [ ]:
pd.read_sql("""
SELECT 'customers' AS table_name, COUNT(*) AS total_rows FROM customers
UNION ALL
SELECT 'vehicles', COUNT(*) FROM vehicles
UNION ALL
SELECT 'service_records', COUNT(*) FROM service_records
UNION ALL
SELECT 'complaints', COUNT(*) FROM complaints
""", engine)

,table_name,total_rows
0,customers,400
1,vehicles,550
2,service_records,2200
3,complaints,350
